# CHOICE 업서트 노트북

이 노트북은 `modular.db`의 `CHOICE` 테이블에 데이터를 직접 업서트하기 위한 템플릿입니다.

실행 전 확인:
1. `id`는 전역 유일하게 정합니다.
2. `name`은 선택지 세트 이름으로 작성합니다.
3. `contents`는 JSON 구조(dict 또는 list)로 작성합니다.
4. 이후 `ITEM.choice.id`에서 동일한 `CHOICE.id`를 참조합니다.


## CHOICE 테이블 컬럼

- `id`: 선택지 세트 고유 ID
- `name`: 선택지 세트 이름
- `description`: 선택지 세트 설명(nullable)
- `contents`: 선택지 JSON


In [4]:
# 1) 테이블 이름 + 스키마 설정
from pathlib import Path
import re
import sqlite3

DB_PATH = Path(r"C:\Users\user\workspace\2.0-modular\modular.db")
TABLE_NAME = 'CHOICE'

SCHEMA = [
    {'name': 'id', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'name', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'description', 'type': 'TEXT', 'constraints': ''},
    {'name': 'contents', 'type': 'TEXT', 'constraints': 'NOT NULL'},
]
PRIMARY_KEY = ['id']

VALID_TYPES = {'INTEGER', 'REAL', 'TEXT', 'BLOB'}
identifier_re = re.compile(r'^[A-Za-z_][A-Za-z0-9_\.]*$')

def validate_identifier(name: str) -> None:
    if not identifier_re.match(name):
        raise ValueError(f'잘못된 이름: {name} (영문/숫자/언더스코어/점만 허용, 숫자로 시작 불가)')

validate_identifier(TABLE_NAME)
seen = set()
for col in SCHEMA:
    col_name = col['name']
    col_type = col['type'].upper()
    validate_identifier(col_name)
    if col_name in seen:
        raise ValueError(f'중복 컬럼명: {col_name}')
    if col_type not in VALID_TYPES:
        raise ValueError(f'지원하지 않는 타입: {col_type}')
    seen.add(col_name)

print('설정 확인 완료')
print('DB_PATH =', DB_PATH)
print('TABLE_NAME =', TABLE_NAME)
print('SCHEMA =', SCHEMA)
print('PRIMARY_KEY =', PRIMARY_KEY)


설정 확인 완료
DB_PATH = C:\Users\user\workspace\2.0-modular\modular.db
TABLE_NAME = CHOICE
SCHEMA = [{'name': 'id', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'name', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'description', 'type': 'TEXT', 'constraints': ''}, {'name': 'contents', 'type': 'TEXT', 'constraints': 'NOT NULL'}]
PRIMARY_KEY = ['id']


In [5]:
# 2) CHOICE 테이블 구조 확인
conn = sqlite3.connect(DB_PATH)
try:
    cur = conn.execute('PRAGMA table_info("CHOICE")')
    print('table_info(CHOICE)')
    for row in cur.fetchall():
        print(row)
finally:
    conn.close()


table_info(CHOICE)
(0, 'id', 'TEXT', 1, None, 1)
(1, 'name', 'TEXT', 1, None, 0)
(2, 'description', 'TEXT', 0, None, 0)
(3, 'contents', 'TEXT', 1, None, 0)


## 업서트


In [6]:
# 3) CHOICE 테이블 업서트
import json

insert_columns = []
schema_type_map = {}
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').upper()
    schema_type_map[c_name] = c_type
    if 'PRIMARY KEY' in c_constraints and 'AUTOINCREMENT' in c_constraints:
        continue
    insert_columns.append(c_name)

print('입력 대상 컬럼:', insert_columns)

ROWS = [
    {
        'id': 'CHOICE_LIKERT_5',
        'name': '5점 리커트',
        'description': '공통 5점 리커트 응답 세트',
        'contents': {
            '1': '전혀 그렇지 않다',
            '2': '드물게 그렇다',
            '3': '때때로 그렇다',
            '4': '자주 그렇다',
            '5': '거의 항상 그렇다'
        },
    },
    {
        'id': 'CHOICE_GOLDEN_ADULT_PART1_BIPOLAR_7',
        'name': 'GOLDEN 성인용 문항제시 양극형 7점 선택지',
        'description': 'GOLDEN 성인용에서만 사용되는 선택지로 문항별로 보기 라벨이 달라 별도 세트로 분리되어 있습니다.',
        'contents': {
            '1': {'1': '말을 적게 하는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '말을 많이 하는 편이다.'},
            '2': {'1': '이론에 대해 이야기한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '사실에 대해 이야기한다.'},
            '3': {'1': '독서이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '새로운 사람들을 만나는 것이다.'},
            '4': {'1': '즉각적이고 일단 행동하고 본다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '계획적이고 목표를 세워 진행한다.'},
            '5': {'1': '모든 것이 잘 정돈된 일반적인 정원이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '이국적인 식물들이 있는 독특한 정원이다.'},
            '6': {'1': '그냥 말없이 떠날 것이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '작별인사를 하는 데 많은 시간을 들일 것이다.'},
            '7': {'1': '모든 과제에 대한 목표를 미리 계획하는 것을 좋아한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '하나의 목표를 설정하고, 문제가 생기면 그때 그 문제를 해결하는 것을 좋아한다.'},
            '8': {'1': '그 감정은 비판받아야 한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '그 감정도 이해되어야 한다.'},
            '9': {'1': '상식이 많다는 점이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '상상력이 풍부하다는 점이다.'},
            '10': {'1': '마감 시간이 임박해야 일을 시작하는 사람이라고 말한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '가장 먼저 일을 시작하는 사람이라고 말한다.'},
            '11': {'1': '과학, 기술, 정치, 경제 분야이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '예술, 문학, 음악, 인문 분야이다.'},
            '12': {'1': '계획을 세우고, 이를 지키고자 노력한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '계획에 없었더라도 내가 원하는 것이 생기면 구매한다.'},
            '13': {'1': '타인의 어려움에 잘 공감하는 사람이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '감정적인 상황일지라도 이성적이다.'},
            '14': {'1': '주저한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '기꺼이 하고자 한다.'},
            '15': {'1': '조용히 다른 주제를 생각한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '그 대화에 참여하고 싶어진다.'},
            '16': {'1': '짜증을 낸다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '즐겁게 여긴다.'},
            '17': {'1': '소소한 기쁨들로 채워져 있다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '고통과 역경으로 채워져 있다.'},
            '18': {'1': '방해가 되지 않는다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '다른 사람들에 비해 더 방해가 되는 편이다.'},
            '19': {'1': '내가 생각한 방향이 맞다고 일행을 설득하는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '일행의 입장에서 그들이 생각하는 방향으로 가는 편이다.'},
            '20': {'1': '대체로 재미있는 부분으로 건너뛰고 싶어진다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '철저히 준비해야 한다는 압박감을 느낀다.'},
            '21': {'1': '따뜻하게 환영받는 느낌이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '불편하다.'},
            '22': {'1': '불편해하는 사람들을 알아챈다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '다른 사람들에 대해 신경 쓰지 않는다.'},
            '23': {'1': '수줍음이 많고 쉽게 당황한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '다른 사람들을 신경 쓰지 않는다.'},
            '24': {'1': '대화를 주도한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '다른 사람들이 다가와 주기를 기다린다.'},
            '25': {'1': '스스로 목적지를 정하여 여행하는 것을 좋아한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '다른 사람의 안내를 따라 여행하는 것을 좋아한다.'},
            '26': {'1': '실용적이고, 가격 대비 효용이 높으며, 검증된 방식으로 설계하고 싶다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '보기에 멋지게 설계하고 싶다.'},
            '27': {'1': '세세한 부분까지 미리 생각해 둔다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '예상치 못한 상황이 발생하면 그때그때 처리한다.'},
            '28': {'1': '빠르고 재치 있게 잘 대답한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '대화에 적절한 내용을 생각하다가 말할 때를 놓치곤 한다.'},
            '29': {'1': '꾸밈없이 있는 그대로 말하는 사람이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '새롭고 독특한 방식으로 자신의 생각을 표현하는 사람이다.'},
            '30': {'1': '주어진 사실 그 이상으로 생각하지 않으려고 조심한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '대략적이라도 추이를 찾아 해석하려고 한다.'},
            '31': {'1': '여행 계획을 세우는 데에 많은 시간을 쓰는 것을 좋아하지 않는다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '여행을 가는 것만큼 계획 세우는 것도 즐긴다.'},
            '32': {'1': '문제를 걱정하는 데 많은 시간을 쓰지 않는다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '문제를 걱정하는 데 많은 시간을 쓴다.'},
            '33': {'1': '천천히 사귀는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '빨리 사귀는 편이다.'},
            '34': {'1': '그 사람의 주장에서 허점을 찾는 데 더 집중하는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '칭찬거리를 찾는 데 더 집중하는 편이다.'},
            '35': {'1': '기분 나쁘다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '신경 쓰지 않는다.'},
            '36': {'1': '기분이 좋고 걱정할 만한 일이 없었다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '우울하고 걱정되는 일이 꽤 있었다.'},
            '37': {'1': '읽지 않고 그냥 놔둔다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '즉시 분류하고, 원하지 않는 메일은 지운다.'},
            '38': {'1': '그 결정이 내 주변 사람이나 나에게 미칠 영향을 고려한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '객관적으로 정보를 수집하고 분석한다.'},
            '39': {'1': '효과가 있었던 기존의 방법을 찾는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '새롭고 더 나은 방법을 찾아보는 편이다.'},
            '40': {'1': '열정적으로 내 의견을 표현한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '논리적이고 냉정하게 대처한다.'},
            '41': {'1': '여러 사람들과 어울리는 모임에 나가거나 쇼핑하는 것을 즐긴다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '집에서 TV를 보거나 독서를 하며 조용히 저녁시간을 보낸다.'},
            '42': {'1': '여러 일들을 미리 정해놓는 것을 좋아한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '계획을 바꾸거나 사람들을 추가할 여지를 남겨두는 것을 좋아한다.'},
            '43': {'1': '항상 바로 납부한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '가끔 잊어버리거나 늦게 납부한다.'},
            '44': {'1': '내 주변 사람들과 함께 일하는 곳이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '혼자서 일하는 곳이다.'},
            '45': {'1': '논리적인 사람으로 알려져 있다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '정이 많은 사람으로 알려져 있다.'},
            '46': {'1': '즐겁다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '불편하다.'},
            '47': {'1': '먼저 말을 하려 하지 않는다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '먼저 말을 하는 편이다.'},
            '48': {'1': '허황된 상상을 하곤 한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '확고하게 현실적인 편이다.'},
            '49': {'1': '잘 울지 않는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '감동적인 장면에서 눈물을 흘리는 편이다.'},
            '50': {'1': '설명서를 읽어보고 순서에 따른다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '먼저 조립하다가 필요하면 설명서를 참고한다.'},
            '51': {'1': '물건을 잘못 산 것 같아 자주 후회한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '이보다 더 좋은 결정은 없었다고 행복해 한다.'},
            '52': {'1': '사람들이 나를 좋아하지 않는다고 느낀다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '다른 사람들처럼 나도 좋은 사람이라고 느낀다.'},
            '53': {'1': '아름다운 은유를 사용한 시이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '유명한 역사적 사건에 대한 상세한 설명이다.'},
            '54': {'1': '그들의 감정에 쉽게 휩쓸린다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '거리를 두고 평정심을 유지한다.'},
            '55': {'1': '세세하게 살피는 꼼꼼한 사람이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '그보다는 더 크게 보는 사람이다.'},
            '56': {'1': '미래에 대하여 희망적으로 느꼈다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '우울했다.'},
            '57': {'1': '평화롭고 조용하며 많은 일들이 일어나지 않는 곳이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '볼거리와 할 것들이 많은 곳이다.'},
            '58': {'1': '뛰어날 정도로 능력있는 사람이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '진정성이 느껴지는 진실된 사람이다.'},
            '59': {'1': '주변 사람의 감정에 영향을 받지 않으려 한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '주변 사람의 감정을 배려하려고 한다.'},
            '60': {'1': '교통체증 때문에 늦을 것만 같은 상황에도 평소 시간대로 출발한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '늦을 경우를 대비하여 일찍 출발한다.'},
            '61': {'1': '반드시 가져가야 하는 물건의 목록을 만들고 잊지 않으려 한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '기억나는대로 많은 것을 챙긴다. 깜빡한 물건들은 여행지에서 대체품을 사거나 없이 지낸다.'},
            '62': {'1': '비관적이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '낙관적이다.'},
            '63': {'1': '전에 시도 해보지 않았던 새로운 생각을 한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '기존의 방법을 적용하는 것을 좋아한다.'},
            '64': {'1': '변화가 없으면 지루하기에 많은 것을 변화시키는 것을 좋아한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '많은 변화가 힘들게 느껴지기 때문에 안정적인 것이 좋다.'},
            '65': {'1': '상황에 맞게 스케줄을 변화시키는 사람이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '스케줄을 그대로 지키는 사람이다.'},
            '66': {'1': '다른 팬들과 함께 경기장에서 보는 것이 좋다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '텔레비전으로 집에서 보는 것이 좋다.'},
            '67': {'1': '다른 사람에게 친절하게 대해 주었을 때이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '다른 사람들이 어려워하는 문제를 해결해 주었을 때다.'},
            '68': {'1': '일이 반복되어도 괜찮다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '일이 반복되면 쉽게 지루함을 느낀다.'},
            '69': {'1': '걱정된다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '걱정할 필요가 없고, 오늘을 즐기며 산다.'},
            '70': {'1': '내 상황이 나아지고 있다고 생각한다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '더 나아질 수 없다고 생각한다.'},
            '71': {'1': '어렵다. 늦지 않으려 애쓰지만 보통 몇 분씩 늦는다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '어렵지 않다. 일찍 도착하려고 애쓴다.'},
            '72': {'1': '다른 사람들 사이에서 돋보이는 것이다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '다른 사람들과 어울리는 것이다.'},
            '73': {'1': '직관을 믿는다.', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '꼼꼼히 따져 살펴본다.'}
        }
    },
    {
        'id': 'CHOICE_GOLDEN_ADULT_PART2_BIPOLAR_7',
        'name': 'GOLDEN 성인용 양극형 7점 선택지',
        'description': 'GOLDEN 성인용에서만 사용되는 선택지로 문항별로 보기 라벨이 달라 별도 세트로 분리되어 있습니다.',
        'contents': {
            '132': {'1': '외향적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '내성적'},
            '133': {'1': '이성적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '감성적'},
            '134': {'1': '동적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '정적'},
            '135': {'1': '혼자 있기 좋아하는', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '어울리기 좋아하는'},
            '136': {'1': '따뜻한', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '분석적'},
            '137': {'1': '조심성 있는', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '과감한'},
            '138': {'1': '직관적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '사실적'},
            '139': {'1': '산만한', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '집중력 있는'},
            '140': {'1': '이상적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '실제적'},
            '141': {'1': '말이 많은', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '말이 적은'},
            '142': {'1': '감정적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '감정적이지 않은'},
            '143': {'1': '임기응변적인', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '초지일관하는'},
            '144': {'1': '부산스러운', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '차분한'},
            '145': {'1': '관습적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '변화적'},
            '146': {'1': '협조적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '고집이 센'},
            '147': {'1': '세부적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '포괄적'},
            '148': {'1': '보살피는', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '경쟁적'},
            '149': {'1': '정해진 것을 따르는', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '변화를 수용하는'},
            '150': {'1': '객관적인', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '인간적인'},
            '151': {'1': '활동적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '생각이 깊은'},
            '152': {'1': '감성적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '감성적이지 않은'},
            '153': {'1': '타협적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '타협적이지 않은'},
            '154': {'1': '즉흥적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '신중한'},
            '155': {'1': '문자 그대로', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '비유적'},
            '156': {'1': '정확한', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '대충하는'},
            '157': {'1': '과묵한', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '표현하는'},
            '158': {'1': '자신감 있는', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '자신감 없는'},
            '159': {'1': '미래지향적인', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '전통을 따르는'},
            '160': {'1': '보이는 것을 중시하는', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '생각을 중시하는'},
            '161': {'1': '곧이 곧대로', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '수완이 좋은'},
            '162': {'1': '즉흥적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '정확한'},
            '163': {'1': '참여적', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '성찰적'},
            '164': {'1': '관습을 따르는', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '관습을 따르지 않는'},
            '165': {'1': '대담한', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '수줍어하는'},
            '166': {'1': '수용적인', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '의심이 많은'},
            '167': {'1': '기분이 좋은', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '긴장한'},
            '168': {'1': '무뚝뚝한', '2': '', '3': '', '4': '', '5': '', '6': '', '7': '상냥한'}
        }
    },
    {
        'id': 'CHOICE_GOLDEN_CHILD_PART1_BIPOLAR_7',
        'name': 'GOLDEN 아동용 문항제시 양극형 7점 선택지',
        'description': 'GOLDEN 아동용에서만 사용되는 선택지로 문항별로 보기 라벨이 달라 별도 문항별 세트로 분리되어 있습니다.',
        'contents': {
            '1': {'1': '말을 많이 하는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '말을 적게 하는 편이다.'},
            '2': {'1': '상식이다.', '2': '', '3': '', '4': '', '5': '', '6': '상상력이다.'},
            '3': {'1': '먼저 그 이유가 알고 싶다.', '2': '', '3': '', '4': '', '5': '', '6': '먼저 그 어려움에 공감한다.'},
            '4': {'1': '세세한 부분까지 미리 생각해 둔다.', '2': '', '3': '', '4': '', '5': '', '6': '예상치 못한 상황이 발생하면 그때그때 처리한다.'},
            '5': {'1': '그 문제가 심각해질까 봐 염려한다.', '2': '', '3': '', '4': '', '5': '', '6': '어떻게든 해결될 것이라고 기대한다.'},
            '6': {'1': '적게 한다.', '2': '', '3': '', '4': '', '5': '', '6': '많이 한다.'},
            '7': {'1': '아이디어를 만드는 것이다.', '2': '', '3': '', '4': '', '5': '', '6': '실제적인 사실을 다루는 것이다.'},
            '8': {'1': '울지 않는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '우는 편이다.'},
            '9': {'1': '계획에 따라 하는 것이 좋다.', '2': '', '3': '', '4': '', '5': '', '6': '상황에 따라 하는 것이 좋다.'},
            '10': {'1': '문제를 걱정하는 데 많은 시간을 쓴다.', '2': '', '3': '', '4': '', '5': '', '6': '문제를 크게 걱정하지 않는 편이다.'},
            '11': {'1': '가족이나 친구랑 있는 것이다.', '2': '', '3': '', '4': '', '5': '', '6': '혼자 있는 것이다.'},
            '12': {'1': '실용적이고 쓸모 있는 것이다.', '2': '', '3': '', '4': '', '5': '', '6': '독특하고 창의적인 것이다.'},
            '13': {'1': '논리와 근거이다.', '2': '', '3': '', '4': '', '5': '', '6': '감정과 기분이다.'},
            '14': {'1': '챙길 물건의 목록을 만든다.', '2': '', '3': '', '4': '', '5': '', '6': '그냥 짐을 싸기 시작한다.'},
            '15': {'1': '걱정된다.', '2': '', '3': '', '4': '', '5': '', '6': '걱정할 필요가 없고, 오늘을 즐기며 산다.'},
            '16': {'1': '좋아한다.', '2': '', '3': '', '4': '', '5': '', '6': '피하게 된다.'},
            '17': {'1': '실제로 도움이 되는 것이다.', '2': '', '3': '', '4': '', '5': '', '6': '창의적인 것이다.'},
            '18': {'1': '위로하고 격려해 준다.', '2': '', '3': '', '4': '', '5': '', '6': '다음에 어떻게 더 잘 볼지 알려준다.'},
            '19': {'1': '꼼꼼히 따져 살펴본다.', '2': '', '3': '', '4': '', '5': '', '6': '나의 직관을 믿는다.'},
            '20': {'1': '수줍음이 많고 쉽게 당황하는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '활발하고 자신감이 있는 편이다.'},
            '21': {'1': '혼자 시간을 보낸다.', '2': '', '3': '', '4': '', '5': '', '6': '사람들을 만난다.'},
            '22': {'1': '대부분의 사람들이 하는 방식을 좋아한다.', '2': '', '3': '', '4': '', '5': '', '6': '나만의 독특한 방식을 좋아한다.'},
            '23': {'1': '논리적인 사람이다.', '2': '', '3': '', '4': '', '5': '', '6': '따뜻한 사람이다.'},
            '24': {'1': '신중하며 조심성이 있는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '즉흥적이고 위험을 감수하는 편이다.'},
            '25': {'1': '신경을 쓰는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '신경 쓰지 않는 편이다.'},
            '26': {'1': '주로 대화를 주도한다.', '2': '', '3': '', '4': '', '5': '', '6': '다른 사람이 말 걸어 주기를 기다린다.'},
            '27': {'1': '일반적이고 평범한 것이다.', '2': '', '3': '', '4': '', '5': '', '6': '새롭고 독특한 것이다.'},
            '28': {'1': '상대방의 단점을 잘 찾는다.', '2': '', '3': '', '4': '', '5': '', '6': '상대방의 장점을 잘 찾는다.'},
            '29': {'1': '정해진 일정을 잘 따르는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '나만의 일정을 만드는 편이다.'},
            '30': {'1': '쉽다.', '2': '', '3': '', '4': '', '5': '', '6': '어렵다.'},
            '31': {'1': '안정적이고 익숙한 일을 좋아한다.', '2': '', '3': '', '4': '', '5': '', '6': '도전적이고 새로운 일을 좋아한다.'},
            '32': {'1': '능력 있고 이성적인 사람이다.', '2': '', '3': '', '4': '', '5': '', '6': '동정심이 많고 따뜻한 사람이다.'},
            '33': {'1': '정해진 규칙을 잘 따르는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '나만의 규칙을 만드는 편이다.'},
            '34': {'1': '다양한 활동에 참여하는 것을 좋아한다.', '2': '', '3': '', '4': '', '5': '', '6': '조용히 생각하는 것을 좋아한다.'},
            '35': {'1': '많은 사람이 인정하는 일반적인 방식이 좋다.', '2': '', '3': '', '4': '', '5': '', '6': '인정받지 못해도 새로운 방식이 좋다.'},
            '36': {'1': '원칙을 따른다.', '2': '', '3': '', '4': '', '5': '', '6': '다른 사람의 입장을 생각한다.'},
            '37': {'1': '제시된 해결책을 따르는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '나만의 해결책을 만드는 편이다.'},
            '38': {'1': '즐겁다.', '2': '', '3': '', '4': '', '5': '', '6': '불편하다.'},
            '39': {'1': '사람을 사실 그대로 대하려고 한다.', '2': '', '3': '', '4': '', '5': '', '6': '사람의 생각과 감정을 이해하려고 한다.'},
            '40': {'1': '마감 기한 전부터 최선을 다한다.', '2': '', '3': '', '4': '', '5': '', '6': '마감 기한이 다가올 때 최선을 다한다.'},
            '41': {'1': '집중해서 혼자 하는 과제이다.', '2': '', '3': '', '4': '', '5': '', '6': '친구들과 의논하며 하는 과제이다.'},
            '42': {'1': '기한보다 일을 일찍 시작하는 편이다.', '2': '', '3': '', '4': '', '5': '', '6': '기한에 임박하여 일을 시작하는 편이다.'},
            '43': {'1': '미루지 않고 바로 한다.', '2': '', '3': '', '4': '', '5': '', '6': '미루다가 마지막에 한다.'}
        }
    },
    {
        'id': 'CHOICE_GOLDEN_CHILD_PART2_BIPOLAR_7',
        'name': 'GOLDEN 아동용 문항제시 양극형 7점 선택지',
        'description': 'GOLDEN 아동용에서만 사용되는 선택지로 문항별로 보기 라벨이 달라 별도 문항별 세트로 분리되어 있습니다.',
        'contents': {
            '44': {'1': '겉으로 표현하는', '2': '', '3': '', '4': '', '5': '', '6': '속으로 생각하는'},
            '45': {'1': '상상력이 많은', '2': '', '3': '', '4': '', '5': '', '6': '현실적인'},
            '46': {'1': '논리적인', '2': '', '3': '', '4': '', '5': '', '6': '감성적인'},
            '47': {'1': '계획에 맞춰 하는', '2': '', '3': '', '4': '', '5': '', '6': '상황에 맞춰 하는'},
            '48': {'1': '걱정하는', '2': '', '3': '', '4': '', '5': '', '6': '여유 있는'},
            '49': {'1': '말이 많은', '2': '', '3': '', '4': '', '5': '', '6': '말이 적은'},
            '50': {'1': '상상력을 발휘하는', '2': '', '3': '', '4': '', '5': '', '6': '상식을 활용하는'},
            '51': {'1': '감정이 풍부한', '2': '', '3': '', '4': '', '5': '', '6': '감정적이지 않은'},
            '52': {'1': '즉흥적으로 행동하는', '2': '', '3': '', '4': '', '5': '', '6': '신중하게 행동하는'},
            '53': {'1': '확신하는', '2': '', '3': '', '4': '', '5': '', '6': '자신 없는'},
            '54': {'1': '많은 친구', '2': '', '3': '', '4': '', '5': '', '6': '소수의 친한 친구'},
            '55': {'1': '유용한 방법', '2': '', '3': '', '4': '', '5': '', '6': '새로운 방법'},
            '56': {'1': '다정한', '2': '', '3': '', '4': '', '5': '', '6': '냉정한'},
            '57': {'1': '조심스럽게 행동하는', '2': '', '3': '', '4': '', '5': '', '6': '거침없이 행동하는'},
            '58': {'1': '조심스러운', '2': '', '3': '', '4': '', '5': '', '6': '자신 있는'},
            '59': {'1': '활동적인', '2': '', '3': '', '4': '', '5': '', '6': '조용한'},
            '60': {'1': '효과적인 아이디어', '2': '', '3': '', '4': '', '5': '', '6': '참신한 아이디어'},
            '61': {'1': '경쟁하는', '2': '', '3': '', '4': '', '5': '', '6': '보살피는'},
            '62': {'1': '정해진 목표', '2': '', '3': '', '4': '', '5': '', '6': '나만의 목표'},
            '63': {'1': '대담한', '2': '', '3': '', '4': '', '5': '', '6': '수줍어하는'},
            '64': {'1': '해왔던 방식', '2': '', '3': '', '4': '', '5': '', '6': '새로운 방식'},
            '65': {'1': '양보하는', '2': '', '3': '', '4': '', '5': '', '6': '양보하지 않는'},
            '66': {'1': '꾸준하게 노력하는', '2': '', '3': '', '4': '', '5': '', '6': '한 번에 최선을 다하는'},
            '67': {'1': '활동이 많은', '2': '', '3': '', '4': '', '5': '', '6': '생각이 많은'},
            '68': {'1': '새로운 방법을 찾는', '2': '', '3': '', '4': '', '5': '', '6': '기존의 방법을 적용하는'},
            '69': {'1': '규칙이 지켜지는 사회', '2': '', '3': '', '4': '', '5': '', '6': '서로 배려하는 사회'},
            '70': {'1': '안정적인', '2': '', '3': '', '4': '', '5': '', '6': '변화하는'},
            '71': {'1': '독립적인', '2': '', '3': '', '4': '', '5': '', '6': '함께하는'},
            '72': {'1': '변화를 싫어하는', '2': '', '3': '', '4': '', '5': '', '6': '변화를 좋아하는'}
        }
    },
    {
        'id': 'CHOICE_LIKERT_7',
        'name': '7점 리커트',
        'description': None,
        'contents': {
            '1': '전혀 나 같지 않음',
            '2': '나 같지 않음',
            '3': '약간 나 같지 않음',
            '4': '나와 같지도 다르지도 않음',
            '5': '약간 나 같음',
            '6': '나 같음',
            '7': '매우 나 같음'
        }
    }
]

def normalize_value(col_name: str, value):
    col_type = schema_type_map[col_name]
    if col_type == 'TEXT' and isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

expected_keys = set(insert_columns)
normalized_rows = []
for i, row in enumerate(ROWS, start=1):
    if set(row.keys()) != expected_keys:
        raise ValueError(
            f'{i}번째 ROW 키 불일치: expected={sorted(expected_keys)}, got={sorted(row.keys())}. '
            'SCHEMA 컬럼명과 ROW 키 이름을 정확히 맞추세요.'
        )
    normalized = {k: normalize_value(k, row[k]) for k in insert_columns}
    normalized_rows.append(normalized)

placeholders = ', '.join(['?'] * len(insert_columns))
col_sql = ', '.join([f'"{c}"' for c in insert_columns])
conflict_columns = ['id']
update_columns = [c for c in insert_columns if c not in conflict_columns]
conflict_sql = ', '.join([f'"{c}"' for c in conflict_columns])
update_sql = ', '.join([f'"{c}" = excluded."{c}"' for c in update_columns])
insert_sql = (
    f'INSERT INTO "{TABLE_NAME}" ({col_sql}) VALUES ({placeholders}) '
    f'ON CONFLICT ({conflict_sql}) DO UPDATE SET {update_sql}'
)
values = [tuple(row[c] for c in insert_columns) for row in normalized_rows]

print('실행 SQL:', insert_sql)
print('입력 건수:', len(values))

conn = sqlite3.connect(DB_PATH)
try:
    conn.executemany(insert_sql, values)
    conn.commit()
    print(f'업서트 완료: {len(values)}건')
finally:
    conn.close()

입력 대상 컬럼: ['id', 'name', 'description', 'contents']
실행 SQL: INSERT INTO "CHOICE" ("id", "name", "description", "contents") VALUES (?, ?, ?, ?) ON CONFLICT ("id") DO UPDATE SET "name" = excluded."name", "description" = excluded."description", "contents" = excluded."contents"
입력 건수: 6
업서트 완료: 6건


In [ ]:
# 4) 결과 조회
conn = sqlite3.connect(DB_PATH)
try:
    cur = conn.execute('SELECT * FROM "CHOICE" ORDER BY "id"')
    rows = cur.fetchall()
    print(f'총 {len(rows)}건')
    for row in rows:
        print(row)
finally:
    conn.close()

총 6건
('CHOICE_GOLDEN_ADULT_PART1_BIPOLAR_7', 'GOLDEN 성인용 문항제시 양극형 7점 선택지', 'GOLDEN 성인용에서만 사용되는 선택지로 문항별로 보기 라벨이 달라 별도 세트로 분리되어 있습니다.', '{"1": {"1": "말을 적게 하는 편이다.", "2": "", "3": "", "4": "", "5": "", "6": "", "7": "말을 많이 하는 편이다."}, "2": {"1": "이론에 대해 이야기한다.", "2": "", "3": "", "4": "", "5": "", "6": "", "7": "사실에 대해 이야기한다."}, "3": {"1": "독서이다.", "2": "", "3": "", "4": "", "5": "", "6": "", "7": "새로운 사람들을 만나는 것이다."}, "4": {"1": "즉각적이고 일단 행동하고 본다.", "2": "", "3": "", "4": "", "5": "", "6": "", "7": "계획적이고 목표를 세워 진행한다."}, "5": {"1": "모든 것이 잘 정돈된 일반적인 정원이다.", "2": "", "3": "", "4": "", "5": "", "6": "", "7": "이국적인 식물들이 있는 독특한 정원이다."}, "6": {"1": "그냥 말없이 떠날 것이다.", "2": "", "3": "", "4": "", "5": "", "6": "", "7": "작별인사를 하는 데 많은 시간을 들일 것이다."}, "7": {"1": "모든 과제에 대한 목표를 미리 계획하는 것을 좋아한다.", "2": "", "3": "", "4": "", "5": "", "6": "", "7": "하나의 목표를 설정하고, 문제가 생기면 그때 그 문제를 해결하는 것을 좋아한다."}, "8": {"1": "그 감정은 비판받아야 한다.", "2": "", "3": "", "4": "", "5": "", "6": "", "7": "그 감정도 이해되어야 한다."}, "9": {"